# Lab 1. Embedding Based Machine Translation


## Part 1: Bilingual dictionary induction and unsupervised embedding-based MT


*Примечание: основано на материалах курса [NLP от YSDA](https://github.com/yandexdataschool/nlp_course/).*




*Редакторы оригинала: Nikolay Karpachev, Valery Marchenkov*

**В этом домашнем задании** вы создадите систему машинного перевода **без использования параллельных корпусов**, выравнивания, attention-механизмов, сверхглубоких рекуррентных нейросетей и прочих «суперкрутых» технологий.

Но даже **без параллельных корпусов** такая система может работать достаточно хорошо (по крайней мере, для близкородственных языков).


### Фрагмент списка Сводеша для некоторых славянских языков

Список Сводеша — это инструмент лексикостатистики. Он назван в честь американского лингвиста Морриса Сводеша и содержит базовую лексику. Этот список используется для определения подгрупп языков и степени их родства.

Таким образом, можно заметить определённое сходство слов в разных славянских языках.


| Russian         | Belorussian              | Ukrainian               | Polish             | Czech                         | Bulgarian            |
|-----------------|--------------------------|-------------------------|--------------------|-------------------------------|-----------------------|
| женщина         | жанчына, кабета, баба    | жінка                   | kobieta            | žena                          | жена                  |
| мужчина         | мужчына                  | чоловік, мужчина        | mężczyzna          | muž                           | мъж                   |
| человек         | чалавек                  | людина, чоловік         | człowiek           | člověk                        | човек                 |
| ребёнок, дитя   | дзіця, дзіцёнак, немаўля | дитина, дитя            | dziecko            | dítě                          | дете                  |
| жена            | жонка                    | дружина, жінка          | żona               | žena, manželka, choť          | съпруга, жена         |
| муж             | муж, гаспадар            | чоловiк, муж            | mąż                | muž, manžel, choť             | съпруг, мъж           |
| мать, мама      | маці, матка              | мати, матір, неня, мама | matka              | matka, máma, 'стар.' mateř    | майка                 |
| отец, тятя      | бацька, тата             | батько, тато, татусь    | ojciec             | otec                          | баща, татко           |
| много           | шмат, багата             | багато                  | wiele              | mnoho, hodně                  | много                 |
| несколько       | некалькі, колькі         | декілька, кілька        | kilka              | několik, pár, trocha          | няколко               |
| другой, иной    | іншы                     | інший                   | inny               | druhý, jiný                   | друг                  |
| зверь, животное | жывёла, звер, істота     | тварина, звір           | zwierzę            | zvíře                         | животно               |
| рыба            | рыба                     | риба                    | ryba               | ryba                          | риба                  |
| птица           | птушка                   | птах, птиця             | ptak               | pták                          | птица                 |
| собака, пёс     | сабака                   | собака, пес             | pies               | pes                           | куче, пес             |
| вошь            | вош                      | воша                    | wesz               | veš                           | въшка                 |
| змея, гад       | змяя                     | змія, гад               | wąż                | had                           | змия                  |
| червь, червяк   | чарвяк                   | хробак, черв'як         | robak              | červ                          | червей                |
| дерево          | дрэва                    | дерево                  | drzewo             | strom, dřevo                  | дърво                 |
| лес             | лес                      | ліс                     | las                | les                           | гора, лес             |
| палка           | кій, палка               | палиця                  | patyk, pręt, pałka | hůl, klacek, prut, kůl, pálka | палка, пръчка, бастун |

Однако распределение контекстов в этих языках демонстрирует ещё большее сходство. И мы можем использовать этот факт в своих целях.

## Data

В этом ноутбуке мы будем использовать **предобученные векторы слов — FastText** (оригинальная статья: [https://arxiv.org/abs/1607.04606](https://arxiv.org/abs/1607.04606)).

Вы можете скачать их с [официального сайта](https://fasttext.cc/docs/en/crawl-vectors.html).
Нам понадобятся эмбеддинги для **английского** и **французского** языков.

In [1]:
# Colab setup: install the notebook dependencies before downloading the large embeddings.
%pip install -q gensim scikit-learn nltk

!wget -nc https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
!gzip -d cc.en.300.vec.gz

!wget -nc https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.fr.300.vec.gz
!gzip -d cc.fr.300.vec.gz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 75.6 MB/s eta 0:00:00
--2026-09-21 20:11:19--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.33, 13.249.182.62, 13.249.182.81, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1325960915 (1.2G) [binary/octet-stream]
Saving to: ‘cc.en.300.vec.gz’

cc.en.300.vec.gz    100%[===================>]   1.23G   238MB/s    in 5.4s    

2026-09-21 20:11:25 (235 MB/s) - ‘cc.en.300.vec.gz’ saved [1325960915/1325960915]

--2026-09-21 20:12:12--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.fr.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.33, 13.249.182.81, 13.249.182.62, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.33|:443... connected.
HTTP request sent, awaiting r

После загрузки и распаковки векторов мы сможем открыть их с помощью библиотеки [gensim](https://radimrehurek.com/gensim/):


In [2]:
from gensim.models import KeyedVectors
import numpy as np


en_emb = KeyedVectors.load_word2vec_format("cc.en.300.vec")
fr_emb = KeyedVectors.load_word2vec_format("cc.fr.300.vec")

После того как вы загрузите векторы, вы сможете использовать интерфейс `KeyedVectors`, чтобы получать эмбеддинги слов и/или находить наиболее похожие слова по их векторам:


In [3]:
august_embedding = en_emb["august"]
august_embedding.shape, august_embedding[:5]

((300,), array([-0.0522,  0.0364, -0.1252,  0.0053,  0.0382], dtype=float32))

In [4]:
en_emb.most_similar([august_embedding])

[('august', 0.9999999403953552),
 ('september', 0.8252838850021362),
 ('october', 0.8111193180084229),
 ('june', 0.8050147891044617),
 ('july', 0.797055184841156),
 ('november', 0.788363516330719),
 ('february', 0.7831973433494568),
 ('december', 0.7824540138244629),
 ('january', 0.7743154168128967),
 ('april', 0.7621643543243408)]

Последняя функция также позволяет изменять количество ближайших слов с помощью аргумента `topn`:


In [5]:
en_emb.most_similar([august_embedding], topn=3)

[('august', 0.9999999403953552),
 ('september', 0.8252838850021362),
 ('october', 0.8111193180084229)]

Ещё одна особенность `KeyedVectors` заключается в том, что он позволяет вычислять эмбеддинги сразу для нескольких слов одновременно:


In [6]:
en_emb[["august", "september"]].shape

(2, 300)

Всё вышесказанное справедливо и для эмбеддингов французского языка.


In [7]:
fr_emb.most_similar([fr_emb["aout"]])

[('aout', 1.0),
 ('Aout', 0.8249964118003845),
 ('juillet', 0.8109882473945618),
 ('fevrier', 0.8072442412376404),
 ('septembre', 0.7838520407676697),
 ('août', 0.779176652431488),
 ('juin', 0.7692081332206726),
 ('octobre', 0.7597455382347107),
 ('decembre', 0.7595790028572083),
 ('avril', 0.7390779256820679)]

Однако эмбеддинги французского и английского языков обучались **независимо друг от друга**. Это означает, что **нет очевидной связи между значениями векторов для схожих слов на французском и английском**:


In [8]:
fr_emb.most_similar([en_emb["august"]])

[('2003Pays', 0.23082853853702545),
 ('Montsoriu', 0.22505579888820648),
 ('2015Pays', 0.22218400239944458),
 ('2013Genre', 0.2095685601234436),
 ('AdiCloud', 0.2018650770187378),
 ('Bagua', 0.20061466097831726),
 ('2003Paysans', 0.2001495361328125),
 ('ValenceLa', 0.2001476287841797),
 ('Luddites', 0.19998176395893097),
 ('Guadalquivir', 0.19875513017177582)]

## Translation

Мы создадим простой переводчик, который будет пытаться предсказывать французский эмбеддинг по английскому. Для этого нам понадобится **датасет пар слов**.


In [9]:
def load_word_pairs(filename):
    en_fr_pairs = []
    en_vectors = []
    fr_vectors = []
    with open(filename, "r") as inpf:
        for line in inpf:
            en, fr = line.rstrip().split(" ")
            if en not in en_emb or fr not in fr_emb:
                continue
            en_fr_pairs.append((en, fr))
            en_vectors.append(en_emb[en])
            fr_vectors.append(fr_emb[fr])
    return en_fr_pairs, np.array(en_vectors), np.array(fr_vectors)

Мы будем обучать нашу модель предсказывать эмбеддинг французского слова по эмбеддингу его английского аналога. Для этого мы разделим тренировочные и тестовые данные на английские и французские слова и вычислим соответствующие эмбеддинги, чтобы получить `X` (эмбеддинги английских слов) и `y` (эмбеддинги французских слов).


In [10]:
!wget -O en-fr.train.txt https://raw.githubusercontent.com/girafe-ai/ml-course/23s_nes/homeworks/hw04_umt/en-fr.train.txt
!wget -O en-fr.test.txt https://raw.githubusercontent.com/girafe-ai/ml-course/23s_nes/homeworks/hw04_umt/en-fr.test.txt

--2026-09-21 20:41:18--  https://raw.githubusercontent.com/girafe-ai/ml-course/23s_nes/homeworks/hw04_umt/en-fr.train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 178608 (174K) [text/plain]
Saving to: ‘en-fr.train.txt’

en-fr.train.txt     100%[===================>] 174.42K  --.-KB/s    in 0.02s   

2026-09-21 20:41:19 (8.65 MB/s) - ‘en-fr.train.txt’ saved [178608/178608]

--2026-09-21 20:41:19--  https://raw.githubusercontent.com/girafe-ai/ml-course/23s_nes/homeworks/hw04_umt/en-fr.test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response...

In [11]:
en_fr_train, X_train, Y_train = load_word_pairs("en-fr.train.txt")
en_fr_test, X_test, Y_test = load_word_pairs("en-fr.test.txt")

In [12]:
en_fr_train[33:44]

[('which', 'lesquels'),
 ('which', 'laquelle'),
 ('which', 'lequel'),
 ('also', 'également'),
 ('also', 'aussi'),
 ('also', 'egalement'),
 ('were', 'étaient'),
 ('but', 'mais'),
 ('have', 'avoir'),
 ('have', 'ont'),
 ('one', 'un')]

## Embedding space mapping **<font color='magenta'>2 балла</font>**

Пусть $x_i \in \mathrm{R}^d$ распределённое представление слова $i$ в исходном языке, а $y_i \in \mathrm{R}^d$ векторное представление его перевода. Наша цель — найти такую линейную трансформацию $W$, которая минимизирует евклидово расстояние между $Wx_i$ и $y_i$ для некоторого поднабора эмбеддингов слов. Так мы можем сформулировать так называемую [задачу Прокруста](https://en.wikipedia.org/wiki/Orthogonal_Procrustes_problem):

$$W^*= \arg\min_W \sum_{i=1}^n\|Wx_i - y_i\|_2$$

или эквивалентно

$$W^*= \arg\min_W \|XW^T - Y\|_F$$

где $\|\cdot\|_F$ норма Фробениуса (или Евклидова).

> **Примечание:** во второй формуле $W$ и $x$ поменялись местами. Это происходит потому, что матрица $X$ составлена из объектов $x_i$ в *строках*, а не *столбцах*, т.е. фактически из $x_i^T$. Следовательно, $X \in \mathbb{R}^{N \times D}$, где $N$ - количество слов, а $D$ - размерность эмбеддинга. То же самое верно и для $Y$.

$W^*= \arg\min_W \sum_{i=1}^n\|Wx_i - y_i\|_2$
выглядит как простая **множественная линейная регрессия без смещения**. В `sklearn` можно отключить смещение в `LinearRegression` с помощью аргумента `fit_intercept` (на самом деле они называют смещение *intercept*). Давайте реализуем это на практике.


In [13]:
from sklearn.linear_model import LinearRegression

mapping = LinearRegression(fit_intercept=False)
mapping.fit(X_train, Y_train)

assert mapping.coef_.shape == (X_train.shape[1], X_train.shape[1])


Давайте посмотрим на ближайших соседей вектора слова *"august"* (*"août"* на французском) после линейного преобразования.


In [14]:
august = mapping.predict(en_emb["august"].reshape(1, -1))
fr_emb.most_similar(august)

[('juin', 0.7553410530090332),
 ('aout', 0.7527692317962646),
 ('juillet', 0.7500795125961304),
 ('septembre', 0.748238205909729),
 ('mars', 0.7415983080863953),
 ('octobre', 0.7395485043525696),
 ('novembre', 0.7313360571861267),
 ('février', 0.7296543121337891),
 ('janvier', 0.7272254228591919),
 ('avril', 0.7249918580055237)]

В качестве метрики качества мы будем использовать **precision@top-1, top-5 и top-10**: для каждого преобразованного английского эмбеддинга считаем, сколько правильных соответствий находится среди N ближайших соседей в пространстве французских эмбеддингов.


In [15]:
def precision(pairs, mapped_vectors, topn=1):
    """
    :args:
        pairs = список правильных пар слов [(en_word_0, fr_word_0), ...]
        mapped_vectors = список эмбеддингов после преобразования из пространства эмбеддингов исходного языка в пространство эмбеддингов целевого языка
        topn = количество ближайших соседей в пространстве эмбеддингов целевого языка, из которых выбираем
    :returns:
        precision_val, число с плавающей точкой, общее количество слов, для которых правильный перевод найден среди top N ближайших соседей.
    """
    assert len(pairs) == len(mapped_vectors)
    assert len(pairs) > 0
    total = len(pairs)
    correct = 0
    for i in range(total):
        pair = pairs[i]
        predicted_vector = mapped_vectors[i]

        nearest_words = fr_emb.most_similar([predicted_vector], topn=topn)
        nearest_words = {word for word, _ in nearest_words}
        correct += int(pair[1] in nearest_words)

    return correct / total

In [16]:
assert precision([("august", "aout")], august, topn=5) == 1.0
assert precision([("august", "aout")], august, topn=9) == 1.0
assert precision([("august", "aout")], august, topn=10) == 1.0

Обратите внимание, что наша функция `precision` принимает **списки пар слов**, тогда как у нас данные представлены в виде **DataFrame**. Однако это не проблема: мы можем получить список (точнее, массив `numpy`) пар через свойство `values`.

In [17]:
assert precision(en_fr_test[:100], X_test[:100]) == 0.0
assert precision(en_fr_test[:100], Y_test[:100]) == 1.0

Давайте посмотрим, насколько хорошо работает наша модель.

In [18]:
precision_top1 = precision(en_fr_test[:100], mapping.predict(X_test[:100]), 1)
precision_top5 = precision(en_fr_test[:100], mapping.predict(X_test[:100]), 5)
precision_top10 = precision(en_fr_test[:100], mapping.predict(X_test[:100]), 10)

In [19]:
print(f'Linear mapping precision@1:  {precision_top1:.3f}')
print(f'Linear mapping precision@5:  {precision_top5:.3f}')
print(f'Linear mapping precision@10: {precision_top10:.3f}')

Linear mapping precision@1:  0.380
Linear mapping precision@5:  0.670
Linear mapping precision@10: 0.780


## Making it better (orthogonal Procrustean problem) **<font color='magenta'>2 балла</font>**

Можно показать, что **самосогласованное линейное отображение между семантическими пространствами должно быть ортогональным**.
Мы можем наложить условие ортогональности на трансформацию $W$. Тогда задача примет вид:


$$(W^T)^*= \arg\min_{W^T} \|XW^T - Y\|_F \text{, где: } W^TW = I$$

$$I \text{- единичная матрица}$$


Вместо того чтобы решать ещё одну задачу регрессии, мы можем найти оптимальное ортогональное преобразование с помощью **сингулярного разложения (SVD)**. Оказывается, оптимальная трансформация $W^*$ выражается через компоненты SVD:


$$X^TY=U\Sigma V^T\text{, сингулярное разложение}$$
$$(W^T)^*=UV^T$$

In [20]:
import numpy as np

cross_covariance = X_train.T @ Y_train
U, _, Vt = np.linalg.svd(cross_covariance)
mapping_svd = U @ Vt

assert mapping_svd.shape == (X_train.shape[1], X_train.shape[1])
assert np.allclose(mapping_svd.T @ mapping_svd, np.eye(mapping_svd.shape[0]), atol=1e-5)

Теперь наша переменная `mapping` — это просто **массив numpy**, то есть у него нет метода `predict`.
Однако из приведённых выше формул мы знаем, что предсказание выполняется через **умножение матриц**:


In [21]:
fr_emb.most_similar([np.matmul(en_emb['august'], mapping_svd)])

[('aout', 0.6705766320228577),
 ('juin', 0.6591026186943054),
 ('juillet', 0.6516767740249634),
 ('septembre', 0.6453961730003357),
 ('octobre', 0.6392979025840759),
 ('mars', 0.6334785223007202),
 ('août', 0.6331560015678406),
 ('février', 0.6244350671768188),
 ('novembre', 0.6244062185287476),
 ('avril', 0.6175950765609741)]

Теперь давайте вычислим наши значения precision и посмотрим, **улучшили ли результаты наш «трюк»**.


In [22]:
svd_precision_top1 = precision(en_fr_test[:100], np.matmul(X_test[:100], mapping_svd), 1)
svd_precision_top5 = precision(en_fr_test[:100], np.matmul(X_test[:100], mapping_svd), 5)
svd_precision_top10 = precision(en_fr_test[:100], np.matmul(X_test[:100], mapping_svd), 10)

print(f'SVD mapping precision@1:  {svd_precision_top1:.3f}')
print(f'SVD mapping precision@5:  {svd_precision_top5:.3f}')
print(f'SVD mapping precision@10: {svd_precision_top10:.3f}')

SVD mapping precision@1:  0.360
SVD mapping precision@5:  0.680
SVD mapping precision@10: 0.770


## Unsupervised embedding-based MT **<font color='magenta'>3 балла</font>**

Теперь давайте **построим наш переводчик на основе эмбеддингов слов**!


Теперь давайте переведём эти предложения **слово за словом**.

Перед этим, однако, не забудьте **токенизировать** ваши предложения. Для этого вам может оказаться очень полезным `nltk.tokenize.WordPunctTokenizer`.


In [23]:
from nltk.tokenize import WordPunctTokenizer

def translate(sentence):
    """
    :args:
        sentence - предложение на английском (str)
    :returns:
        translation - предложение на французском (str)

    * найти эмбеддинг каждого английского слова в предложении
    * преобразовать вектор английского эмбеддинга
    * найти ближайшее французское слово и заменить
    """
    tokenizer = WordPunctTokenizer()
    translated = []

    for word in tokenizer.tokenize(sentence):
        if word not in en_emb:
            translated.append(word)
            continue

        mapped_vector = np.matmul(en_emb[word], mapping_svd)
        nearest_word = fr_emb.most_similar([mapped_vector], topn=1)[0][0]
        translated.append(nearest_word)

    return " ".join(translated)

In [24]:
assert translate(".") == "."
assert translate("I walk around Paris") == "je marcher autour Paris"

Теперь вы можете поиграть с вашей моделью и попробовать получить максимально точные переводы.

**Примечание**: одна из основных проблем — это **слова вне словаря** (out-of-vocabulary). Попробуйте придумать разные способы обработки таких слов (например, сначала можно переводить их в специальный токен **UNK**, а потом переходить к более сложным методам). Удачи!


In [25]:
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords, twitter_samples
import re
import string

nltk.download('twitter_samples')
nltk.download('stopwords')

def process_tweet(tweet):
    '''
    Входные данные:
        tweet: строка, содержащая твит
    Выходные данные:
        tweets_clean: список слов, содержащий обработанный твит
    '''
    stemmer = PorterStemmer()
    stopwords_english = stopwords.words('english')
    # удаляем тикеры фондового рынка, например $GE
    tweet = re.sub(r'\$\w*', '', tweet)
    # удаляем старый стиль ретвита "RT"
    tweet = re.sub(r'^RT[\s]+', '', tweet)
    # удаляем гиперссылки
    tweet = re.sub(r'https?:\/\/.*[\r\n]*', '', tweet)
    # удаляем хэштеги
    # убираем только символ #, оставляя слово
    tweet = re.sub(r'#', '', tweet)
    # токенизируем твит
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True,
                               reduce_len=True)
    tweet_tokens = tokenizer.tokenize(tweet)

    tweets_clean = []
    for word in tweet_tokens:
        # если (слово не является стоп-словом и  # удаляем стоп-слова
        #     слово не является знаком пунктуации):  # удаляем пунктуацию
        if word not in string.punctuation:
            tweets_clean.append(word)

            # стемминг слова, при желании раскомментируйте
            # stem_word = stemmer.stem(word)
            # tweets_clean.append(stem_word)

    return " ".join(tweets_clean)

[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Unzipping corpora/twitter_samples.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [26]:
twitter_samples.strings('positive_tweets.json')[10:15]

['#FollowFriday @wncer1 @Defense_gouv for being top influencers in my community this week :)',
 "Who Wouldn't Love These Big....Juicy....Selfies :) - http://t.co/QVzjgd1uFo http://t.co/oWBL11eQRY",
 '@Mish23615351  follow @jnlazts &amp; http://t.co/RCvcYYO0Iq follow u back :)',
 "@jjulieredburn Perfect, so you already know what's waiting for you :)",
 'Great new opportunity for junior triathletes aged 12 and 13 at the Gatorade series! Get your entries in :) http://t.co/of3DyOzML0']

In [27]:
for i in twitter_samples.strings('positive_tweets.json')[10:15]:
    print(i, process_tweet(i), sep='\n\n', end='\n-----------------\n')

#FollowFriday @wncer1 @Defense_gouv for being top influencers in my community this week :)

followfriday for being top influencers in my community this week :)
-----------------
Who Wouldn't Love These Big....Juicy....Selfies :) - http://t.co/QVzjgd1uFo http://t.co/oWBL11eQRY

who wouldn't love these big ... juicy ... selfies :)
-----------------
@Mish23615351  follow @jnlazts &amp; http://t.co/RCvcYYO0Iq follow u back :)

follow
-----------------
@jjulieredburn Perfect, so you already know what's waiting for you :)

perfect so you already know what's waiting for you :)
-----------------
Great new opportunity for junior triathletes aged 12 and 13 at the Gatorade series! Get your entries in :) http://t.co/of3DyOzML0

great new opportunity for junior triathletes aged 12 and 13 at the gatorade series get your entries in :)
-----------------


Ваш перевод:

In [28]:
for i in twitter_samples.strings('positive_tweets.json')[:10]:
    print(i, process_tweet(i), translate(process_tweet(i)), sep='\n\n', end='\n-----------------\n')

#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)

followfriday for being top engaged members in my community this week :)

twitt pour être top engagé membres dans mon communauté cette semaine okk
-----------------
@Lamb2ja Hey James! How odd :/ Please call our Contact Centre on 02392441234 and we will be able to assist you :) Many thanks!

hey james how odd :/ please call our contact centre on 02392441234 and we will be able to assist you :) many thanks

hey christopher comment bizarre :/ veuillez appeler notre contacter centre sur 02392441234 et nous pourra être puisse amener aider vous okk nombreux merci
-----------------
@DespiteOfficial we had a listen last night :) As You Bleed is an amazing track. When are you in Scotland?!

we had a listen last night :) as you bleed is an amazing track when are you in scotland

nous avait un écouter dernière nuit okk comme vous saigner est un incroyable track quand sont vous dans eco

## Выводы

Линейное отображение переводит английские embedding-векторы во французское пространство, но ортогональное SVD-отображение дополнительно сохраняет геометрию пространства. Сравнение `precision@1`, `precision@5` и `precision@10` показывает, насколько это ограничение помогает находить правильные переводы среди ближайших соседей.

Пословный перевод чувствителен к отсутствующим словам, морфологии и контексту. В текущем решении OOV-слова и пунктуация сохраняются без изменения, поэтому перевод остаётся воспроизводимым и не теряет исходные токены.